# Pandas Datetime Handling and Time-Series Fundamentals

## Converting Strings to Datetime Objects

The Date column is usually loaded from a CSV as text (strings).

To perform time-based operations such as:
- sorting by date
- extracting year, month, day, or hour
- filtering by date ranges
- resampling time-series data

the column must be converted to Pandas datetime objects.

`pd.to_datetime()` performs this conversion.

In [ ]:
import pandas as pd

df = pd.read_csv('ETH_1h.csv')
print(df.head())


The Date column in the CSV file uses a custom date format.

When Pandas cannot reliably infer the format automatically, the format should be specified explicitly using the `format` parameter of `pd.to_datetime()`.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format = '%Y-%m-%d %I-%p')
df.loc[0, 'Date'].day_name()

### Or
Instead of converting dates after loading the data, they can be parsed directly during import using the `parse_dates` parameter.

In [ ]:
df = pd.read_csv('ETH_1h.csv', parse_dates=['Date'], date_format='%Y-%m-%d %I-%p')
df.head()

## Get day names from dates of a whole series

The `.dt` accessor provides vectorized datetime operations for a Pandas Series containing datetime values. Methods such as `day_name()`, `month_name()`, `year`, and `hour` can be applied to the entire Series through `.dt`.

In [ ]:
df['Date'].dt.day_name()

## Dates Can Be Compared Like Numbers

### Question
Find the earliest date

In [ ]:
df['Date'].min()

Find the latest date

In [ ]:
df['Date'].max()

Find the total number of days (timedelta)

In [ ]:
df['Date'].max()-df['Date'].min()

## Filtering by Dates

### Question
Get rows from 2019

In [ ]:
filt = (df['Date']>='2019') & (df['Date']<'2020')
df[filt]

We can also use datetime object for comparison aswell

In [ ]:
filt = (df['Date']>=pd.to_datetime('2019-01-01')) & (df['Date']<pd.to_datetime('2020-01-01'))
df[filt]

## Setting a Column as the Index

The `set_index()` method replaces the default index with a specified column.

For time-series datasets, the Date column is commonly used as the index because it enables convenient date-based operations such as:

- Selecting rows by date
- Filtering date ranges
- Resampling data
- Time-series analysis

### Syntax

```python
df.set_index('Date', inplace=True)
```

### Parameters

- `'Date'` : The column to be used as the new index.
- `inplace=True` : Modifies the original DataFrame directly instead of returning a new DataFrame.

In [ ]:
df.set_index('Date', inplace=True)
df

Get rows from 2019

In [ ]:
df.loc['2019']

### Question
Get mean closing price from january and february of 2020

In [ ]:
df.sort_index(inplace=True)
df.loc['2020-01':'2020-02']['Close'].mean()

### Question
What was the highest 'High' value of January 26th of 2020

In [ ]:
df.loc['2020-01-26']['High'].max()

## Resampling Time-Series Data

Time-series data is often recorded at a higher frequency than needed for analysis.

In this dataset, each row represents one hour of trading activity.

The `resample()` method groups data into larger time intervals based on the DatetimeIndex.

In [ ]:
df['High'].resample('D').max()

### Question
How can we find the highest ETH price recorded on day '2020-01-26' using hourly trading data?

In [ ]:
highs = df['High'].resample('D').max()
highs['2020-01-26']

### Question
After calculating the daily highest ETH prices, how can Matplotlib be used to create a labeled line chart for visualization

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(highs)
plt.title('Daily Highest ETH Price')
plt.xlabel("Date")
plt.ylabel('Highs')

## Aggregating Multiple Columns During Resampling

The `agg()` method allows different aggregation functions to be applied to different columns in a single operation.

This is useful when each column requires a different summary statistic.

For weekly ETH data:

- Open price → first value of the week
- Close price → last value of the week
- High price → maximum value of the week
- Low price → minimum value of the week
- Volume → total traded volume during the week

The result is a DataFrame containing one row per week with summarized trading information.

### Question

How can different aggregation functions be applied to different columns while resampling hourly ETH data into weekly summaries?

In [ ]:
df.resample('W').agg({'Open':'first', 'Close':'last', 'High':'max', 'Low':'min', 'Volume':'sum'})